# Proyecto integrador — Exploración y limpieza de datos (Online Retail II)

**Autor:** Francisco Ramírez  
**Fuente:** Chen, D. (2012). *Online Retail II* [conjunto de datos]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D (licencia CC BY 4.0).  
**Caso de negocio:** pronóstico de demanda y apoyo a la reposición de inventario para un mayorista británico ficticio de artículos de regalo.

**Objetivo:** revisar la calidad de los datos, aplicar reglas de limpieza justificadas y calcular los indicadores que sustentan el Business Case del Hito I, incluidos la estacionalidad, la concentración de ventas por producto y el patrón semanal de demanda.

**Transparencia sobre el uso de IA.** El análisis y las decisiones finales son responsabilidad del estudiante. ChatGPT y Claude se utilizaron como apoyo para revisar el método, detectar riesgos y proponer mejoras. Las celdas identificadas con 🔵 **[IA]** señalan ajustes o criterios sugeridos con esa asistencia.


<div style="border-left: 5px solid #2878B5; padding: 8px 12px; background: #EAF4FF; margin: 8px 0;">
<strong style="color:#185A8D">🔵 [IA]</strong> Ajuste metodológico o comentario sugerido durante la revisión con herramientas de IA. El estudiante debe validarlo y poder explicarlo.
</div>
<div style="border-left: 5px solid #2E8B57; padding: 8px 12px; background: #EAF7EF; margin: 8px 0;">
<strong style="color:#216E45">🟢 🟢 [COMENTARIO]</strong> Explicación general del procedimiento; la marca no indica una autoría independiente.
</div>
<p><em>Nota:</em> Jupyter aplica el color de los comentarios según el tema del editor; por eso, dentro del código se usan las marcas 🔵 [IA] para distinguir los cambios, y esta leyenda utiliza colores para facilitar la lectura.</p>

## 0. Configuración

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# 🔵 [IA] Se revisan varias ubicaciones para que el notebook funcione tanto en el repositorio como en una carpeta local.
# 🟢 Guarda el Excel original en data/raw/ y no lo incluyas en el historial de Git.
RUTAS_CANDIDATAS = [
    Path("data/raw/online_retail_II.xlsx"),
    Path("data/raw/Online Retail II.xlsx"),
    Path("online_retail_II.xlsx"),
    Path("Online Retail II.xlsx"),
    Path("../data/raw/online_retail_II.xlsx"),
]
RUTA_EXCEL = next((ruta.resolve() for ruta in RUTAS_CANDIDATAS if ruta.is_file()), None)

if RUTA_EXCEL is None:
    candidatas = "\n".join(f"- {ruta}" for ruta in RUTAS_CANDIDATAS)
    raise FileNotFoundError(
        "No se encontró online_retail_II.xlsx. Coloca el archivo en data/raw/ "
        "o junto al notebook. Rutas revisadas:\n" + candidatas
    )

print(f"Archivo de datos: {RUTA_EXCEL}")

Archivo de datos: C:\Users\frank\Reabastece360\data\raw\online_retail_II.xlsx


## 1. Carga de los datos

El libro de Excel trae **dos hojas, una por año** (2009-2010 y 2010-2011). Las leo y las uno en un solo DataFrame.

Las hojas se traslapan del **1 al 9 de diciembre de 2010**. Eliminaré filas duplicadas tras normalizar Invoice y StockCode, pero no asumiré que una cancelación corresponde a una venta concreta solo porque coincidan cliente, producto, cantidad y precio: el archivo no ofrece una llave fiable de relación entre ambas líneas.


In [2]:
# 🔵 [IA] Leo Invoice y StockCode como texto desde el inicio para mantener consistentes los códigos.
sheets_dict = pd.read_excel(
    RUTA_EXCEL,
    sheet_name=None,
    dtype={"Invoice": str, "StockCode": str},
)

for nombre_hoja, tabla_hoja in sheets_dict.items():
    print(f"Hoja: {nombre_hoja}")
    print(f"Filas y columnas: {tabla_hoja.shape}")

# 🟢 Uno las hojas y creo un índice nuevo para evitar que se repitan los índices originales.
df = pd.concat(sheets_dict.values(), ignore_index=True)
print(f"Tamaño combinado: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")

Hoja: Year 2009-2010
Filas y columnas: (525461, 8)
Hoja: Year 2010-2011
Filas y columnas: (541910, 8)
Tamaño combinado: (1067371, 8)
Columnas: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


## 2. Reconocimiento de los datos (antes de limpiar)

Antes de tocar nada quiero saber qué tan sucio está el dato: cuántas filas hay, qué período cubren, dónde faltan valores y qué casos sospechosos existen.

In [3]:
print("\n--- Revisión inicial de los datos ---")

# 🟢 1. Cantidad total de filas.
print(f"1. Total rows: {len(df):,}")

# 🟢 2. Período cubierto por las transacciones.
print(f"2. Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

# 🟢 3. Valores ausentes. Algunas compras no tienen un cliente registrado y siguen siendo ventas válidas.
# 🟢 Por eso, no se eliminan filas solo porque falte el identificador del cliente.
print("\n3. Valores ausentes por columna:")
print(df.isna().sum())

# 🟢 4. Facturas cuyo código comienza con C; suelen corresponder a cancelaciones o devoluciones.
cancellations = df["Invoice"].astype(str).str.startswith("C").sum()
print(f"\n4. Facturas que comienzan con C: {cancellations:,}")

# 🟢 5. Filas con cantidad o precio no positivo. Este conteo se solapa con el anterior,
# 🟢 ya que muchas cancelaciones tienen cantidad negativa; los conteos no deben sumarse.
invalid_qty_price = ((df["Quantity"] <= 0) | (df["Price"] <= 0)).sum()
print(f"5. Filas con cantidad o precio no positivo: {invalid_qty_price:,}")

# 🟢 6. Códigos distintos; algunos identifican servicios, como el envío, y no productos.
distinct_stockcodes = df["StockCode"].nunique()
print(f"6. Códigos distintos de producto/servicio: {distinct_stockcodes:,}")


--- Revisión inicial de los datos ---
1. Total rows: 1,067,371
2. Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00

3. Valores ausentes por columna:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

4. Facturas que comienzan con C: 19,494
5. Filas con cantidad o precio no positivo: 25,700
6. Códigos distintos de producto/servicio: 5,305


## 3. Limpieza con reglas justificadas

Aplico las reglas **una por una** y anoto cuántas filas elimina cada una, para dejar un rastro auditable:

| Regla | Por qué |
|---|---|
| 0. Estandarizar textos | Evita que `85123a` y `85123A` cuenten como códigos distintos |
| 1. Duplicados tras normalizar | Evita volver a contar filas idénticas; el solapamiento entre hojas explica parte de ellas |
| 2. Separar cancelaciones | Las facturas que empiezan con `C` se guardan aparte; no se consideran demanda positiva |
| 3. Cantidad o precio no positivo | No representan una venta positiva para el baseline de demanda |
| 4. Filtrar códigos de producto iniciales | Se usa una regla explícita y se imprime una auditoría de los códigos excluidos |

**🔵 Criterio revisado con apoyo de IA:** no descuento automáticamente una venta por una coincidencia de cliente, producto, cantidad y precio. Sin una relación fiable entre líneas, esa regla podría borrar una venta legítima. Por eso, las cancelaciones se informan por separado y las ventas positivas se presentan como una proxy bruta.

**Limitación del filtro de SKU:** el patrón inicial `^\d{5}[A-Z]*$` también puede excluir códigos comerciales válidos no conformes al patrón. La auditoría impresa antes del filtro debe revisarse y las excepciones documentarse antes de la entrega final.


In [4]:
print("\n--- Proceso de limpieza ---")
initial_rows = len(df)

# 🟢 Paso 0: estandarizar los identificadores antes de buscar duplicados.
df["Invoice"] = df["Invoice"].astype(str).str.strip()
df["StockCode"] = df["StockCode"].astype(str).str.strip().str.upper()

# 🟢 Paso 1: eliminar filas idénticas después de normalizar los identificadores.
df_clean = df.drop_duplicates().copy()
removed_dups = initial_rows - len(df_clean)
print(f"1. Filas duplicadas eliminadas tras normalizar: {removed_dups:,}")

# 🔵 [IA] Compruebo cuántas filas duplicadas están en el período compartido por las dos hojas.
en_traslape = (df["InvoiceDate"] >= "2010-12-01") & (df["InvoiceDate"] < "2010-12-10")
dups_traslape = (df.duplicated() & en_traslape).sum()
print(f"   Duplicados en el solapamiento (1–9 dic. 2010): {dups_traslape:,}")
n_filas_tras_dedup = len(df_clean)

# 🔵 [IA] Separo las cancelaciones para analizarlas aparte y evitar tratarlas como demanda positiva.
# 🔵 [IA] No se intenta asociar cada cancelación a una venta concreta sin una llave fiable.
es_cancelacion = df_clean["Invoice"].str.startswith("C", na=False)
df_canc = df_clean.loc[es_cancelacion].copy()
removed_canc = int(es_cancelacion.sum())
df_clean = df_clean.loc[~es_cancelacion].copy()
print(f"2. Filas de cancelación separadas: {removed_canc:,}")

# 🟢 Paso 3: conservar líneas con cantidad y precio estrictamente positivos.
current_len = len(df_clean)
df_clean = df_clean[(df_clean["Quantity"] > 0) & (df_clean["Price"] > 0)].copy()
removed_invalid = current_len - len(df_clean)
print(f"3. Filas descartadas por cantidad/precio no positivo: {removed_invalid:,}")

# # 🟢 Paso 4: revisar los códigos antes de aplicar el filtro inicial de productos.
# patron_producto = r"^\d{5}[A-Z]*$"
# mascara_producto = df_clean["StockCode"].str.match(patron_producto, na=False)
# codigos_excluidos = (
#     df_clean.loc[~mascara_producto]
#     .groupby("StockCode", dropna=False)
#     .agg(Filas=("StockCode", "size"), Descripcion=("Description", "first"))
#     .sort_values("Filas", ascending=False)
# )
# print(f"4. Filas con código no estándar antes del filtro: {int((~mascara_producto).sum()):,}")
# print(f"   Códigos excluidos distintos: {len(codigos_excluidos):,}")
# print("   Códigos excluidos más frecuentes (revisar si son productos o servicios válidos):")
# print(codigos_excluidos.head(20).to_string())

# df_clean = df_clean.loc[mascara_producto].copy()
# removed_codes = current_len - removed_invalid - len(df_clean)  # 🟢 filas quitadas por el patrón, no por qty/precio
# print(f"   Non-standard rows removed by the pattern: {removed_codes:,}; distinct codes: {len(codigos_excluidos):,}")

# # 🔵 [IA] La cifra es una proxy bruta: no se descuentan devoluciones sin una relación fiable con la venta original.
# # 🟢 Se usa solo como referencia descriptiva al comparar el valor de las cancelaciones.
# ventas_brutas = (df_clean["Quantity"] * df_clean["Price"]).sum()
# print(f"\nTamaño final de los datos limpios: {df_clean.shape}")
# print(f"Proxy de ventas positivas brutas (todos los períodos): £{ventas_brutas:,.2f}")


--- Proceso de limpieza ---
1. Filas duplicadas eliminadas tras normalizar: 34,335
   Duplicados en el solapamiento (1–9 dic. 2010): 22,844
2. Filas de cancelación separadas: 19,104
3. Filas descartadas por cantidad/precio no positivo: 6,019


In [5]:
# 🟢 Paso 4: auditar códigos no estándar y aplicar la regla
# de elegibilidad de productos físicos.

patron_producto = r"^\d{5}[A-Z]*$"

# Excepciones auditadas: códigos identificados como mercancía física.
# Se mantienen explícitamente porque no cumplen el patrón estándar.
codigos_excepcion_producto = {
    "DCGS0066N",
    "DCGS0041",
    "DCGS0044",
    "DCGS0058",
    "DCGS0062",
    "DCGS0037",
    "DCGS0068",
    "DCGS0075",
    "DCGS0072",
    "DCGS0003",
    "DCGS0004",
    "DCGS0069",
    "DCGS0070",
    "DCGS0076",
    "DCGSSBOY",
    "DCGSSGIRL",
    "PADS",
    "SP1002",
}

# Normalizar los códigos para compararlos de forma consistente.
# No modifica todavía los valores almacenados en df_clean.
stockcode_norm = (
    df_clean["StockCode"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Identificar códigos que cumplen el patrón habitual.
mascara_patron_estandar = stockcode_norm.str.match(
    patron_producto,
    na=False,
)

# Identificar excepciones aprobadas en la auditoría.
mascara_excepcion_producto = stockcode_norm.isin(
    codigos_excepcion_producto
)

# Auditar todos los códigos que no cumplen el patrón habitual.
auditoria_no_estandar = df_clean.loc[
    ~mascara_patron_estandar,
    ["Description"],
].copy()

auditoria_no_estandar["StockCode_auditado"] = (
    stockcode_norm.loc[~mascara_patron_estandar].to_numpy()
)

codigos_excluidos = (
    auditoria_no_estandar
    .groupby("StockCode_auditado", dropna=False)
    .agg(
        Filas=("Description", "size"),
        Descripcion=("Description", "first"),
    )
    .sort_values("Filas", ascending=False)
)

# Documentar la decisión aplicada a cada código no estándar.
codigos_excluidos["Decision"] = [
    (
        "INCLUIR: excepción de mercancía auditada"
        if str(codigo).strip().upper()
        in codigos_excepcion_producto
        else "EXCLUIR: no elegible para demanda física"
    )
    for codigo in codigos_excluidos.index
]

print("\n--- Auditoría de códigos no estándar ---")
print(
    "Códigos no estándar distintos: "
    f"{len(codigos_excluidos):,}"
)
print(codigos_excluidos.to_string())

# Aceptar códigos estándar O excepciones de mercancía auditadas.
mascara_producto = (
    mascara_patron_estandar
    | mascara_excepcion_producto
)

# Guardar el tamaño anterior para calcular las exclusiones reales.
filas_antes_filtro_producto = len(df_clean)

# Aplicar el filtro sin modificar el DataFrame original.
df_clean = df_clean.loc[mascara_producto].copy()

filas_excluidas_producto = (
    filas_antes_filtro_producto - len(df_clean)
)

filas_retenidas_por_excepcion = int(
    mascara_excepcion_producto.sum()
)

skus_excepcion_presentes = df_clean.loc[
    stockcode_norm.loc[df_clean.index].isin(
        codigos_excepcion_producto
    ),
    "StockCode",
].nunique()

print("\n--- Resultado de elegibilidad de productos ---")
print(
    "Filas retenidas por excepciones auditadas: "
    f"{filas_retenidas_por_excepcion:,}"
)
print(
    "Códigos de excepción presentes tras el filtro: "
    f"{skus_excepcion_presentes:,}"
)
print(
    "Filas excluidas por elegibilidad: "
    f"{filas_excluidas_producto:,}"
)
print(
    f"Tamaño final de los datos limpios: {df_clean.shape}"
)

# 🟢 La cifra es una proxy bruta; no equivale a utilidad.
# Las cancelaciones se analizan por separado porque no siempre
# pueden asociarse de forma fiable con la venta original.
ventas_brutas = (
    df_clean["Quantity"] * df_clean["Price"]
).sum()

print(
    "\nProxy de ventas positivas brutas "
    f"(todos los períodos): £{ventas_brutas:,.2f}"
)


--- Auditoría de códigos no estándar ---
Códigos no estándar distintos: 38
                    Filas                          Descripcion                                  Decision
StockCode_auditado                                                                                      
POST                 1851                              POSTAGE  EXCLUIR: no elegible para demanda física
DOT                  1415                       DOTCOM POSTAGE  EXCLUIR: no elegible para demanda física
M                     851                               Manual  EXCLUIR: no elegible para demanda física
C2                    267                             CARRIAGE  EXCLUIR: no elegible para demanda física
ADJUST                 36  Adjustment by john on 26/01/2010 16  EXCLUIR: no elegible para demanda física
BANK CHARGES           34                         Bank Charges  EXCLUIR: no elegible para demanda física
DCGS0058               30                     MISO PRETTY  GUM  INCLUIR: excepción d

## 4. Ingresos y estacionalidad

Ingreso por línea = cantidad × precio. Trabajo con los **12 meses completos de diciembre de 2010 a noviembre de 2011** (diciembre de 2011 está incompleto: los datos terminan el día 9).

La pregunta clave del negocio: ¿qué parte del año se vende en septiembre-noviembre?

In [6]:
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["Price"]

# 🔵 [IA] Selecciono doce meses completos y creo una copia porque después agregaré columnas a esta tabla;
# 🟢 así evito modificar accidentalmente una vista del conjunto original.
df_12m = df_clean[
    (df_clean["InvoiceDate"] >= "2010-12-01")
    & (df_clean["InvoiceDate"] < "2011-12-01")
].copy()

total_revenue_year = df_12m["Revenue"].sum()
print(f"Ingresos brutos proxy (dic. 2010–nov. 2011): £{total_revenue_year:,.2f}")

# 🟢 Suma de ingresos por mes.
df_12m["YearMonth"] = df_12m["InvoiceDate"].dt.to_period("M")
monthly_revenue = df_12m.groupby("YearMonth")["Revenue"].sum()
print("\nIngresos por mes:")
print(monthly_revenue)

# 🟢 Participación de septiembre a noviembre en el total anual.
sep_nov_revenue = df_12m[df_12m["InvoiceDate"].dt.month.isin([9, 10, 11])]["Revenue"].sum()
sep_nov_share = sep_nov_revenue / total_revenue_year
print(f"\nParticipación de septiembre a noviembre: {sep_nov_share:.1%}")

Ingresos brutos proxy (dic. 2010–nov. 2011): £9,632,728.40

Ingresos por mes:
YearMonth
2010-12     775638.360
2011-01     670380.240
2011-02     507799.870
2011-03     689833.510
2011-04     515377.991
2011-05     739953.000
2011-06     737558.980
2011-07     688219.341
2011-08     724216.500
2011-09    1028320.381
2011-10    1103314.250
2011-11    1452115.980
Freq: M, Name: Revenue, dtype: float64

Participación de septiembre a noviembre: 37.2%


## 5. Análisis ABC (Pareto)

La clasificación ABC ordena los productos por su contribución a los ingresos y ayuda a priorizar el análisis de aquellos con mayor impacto económico.

- **Clase A:** los productos que, ordenados de mayor a menor ingreso, suman el primer 80 % de los ingresos.
- **Clase B:** los que llevan el acumulado del 80 % al 95 %.
- **Clase C:** el resto.

**🔵 Ajuste metodológico revisado con apoyo de IA:** el ABC se calcula sobre los **mismos 12 meses** del Business Case. El producto que cruza el umbral del 80 % se incluye en la clase A, porque contribuye a alcanzar ese acumulado.

In [7]:
def clasificar_abc(df_ventas):
    """Clasifica los productos en A, B o C según su aporte acumulado a los ingresos.

    Parámetros
    ----------
    df_ventas : pandas.DataFrame
        Ventas limpias con las columnas 'StockCode' y 'Revenue'.

    Devuelve
    --------
    pandas.DataFrame
        Un renglón por producto, ordenado de mayor a menor ingreso, con su participación
        acumulada y su clase (A, B o C).
    """
    prod_rev = (
        df_ventas.groupby("StockCode")["Revenue"].sum()
        .sort_values(ascending=False)
        .reset_index()
    )
    total = prod_rev["Revenue"].sum()
    prod_rev["Cumulative_Revenue"] = prod_rev["Revenue"].cumsum()
    prod_rev["Cumulative_Share"] = prod_rev["Cumulative_Revenue"] / total

    # 🔵 [IA] Uso el acumulado previo para incluir en A el producto que cruza el umbral del 80 %.
    # 🔵 [IA] El acumulado previo evita dejar fuera el producto que supera el umbral de cada clase.
    acumulado_previo = prod_rev["Cumulative_Revenue"].shift(fill_value=0) / total
    prod_rev["Clase"] = np.where(acumulado_previo < 0.80, "A",
                         np.where(acumulado_previo < 0.95, "B", "C"))
    return prod_rev


# 🔵 [IA] Mantengo la misma ventana de doce meses que el Business Case para que las cifras sean comparables.
prod_rev = clasificar_abc(df_12m)
# 🔵 [IA] Si una clase no aparece en una prueba o conjunto pequeño, la cuento como cero.
conteo = prod_rev["Clase"].value_counts().reindex(["A", "B", "C"], fill_value=0)
class_a_products = int(conteo["A"])

print("\n--- Clasificación ABC (12 meses: dic. 2010–nov. 2011) ---")
print(f"Total de productos únicos: {len(prod_rev):,}")
print(f"Productos en clase A (hasta cubrir el 80 % de ingresos): {class_a_products:,} ({class_a_products / len(prod_rev):.1%})")
print(f"Productos en clase B: {int(conteo['B']):,} ({conteo['B'] / len(prod_rev):.1%})")
print(f"Productos en clase C: {int(conteo['C']):,} ({conteo['C'] / len(prod_rev):.1%})")

# 🟢 Referencia: el mismo análisis con los dos años completos del dataset (el cálculo original).
prod_rev_total = clasificar_abc(df_clean)
print("\n--- Referencia: conjunto completo (dic. 2009–dic. 2011) ---")
print(f"Total de productos únicos: {len(prod_rev_total):,}")
print(f"Productos en clase A: {(prod_rev_total['Clase'] == 'A').sum():,}")
print(f"Proxy de ingresos del conjunto completo: £{prod_rev_total['Revenue'].sum():,.2f}")


--- Clasificación ABC (12 meses: dic. 2010–nov. 2011) ---
Total de productos únicos: 3,797
Productos en clase A (hasta cubrir el 80 % de ingresos): 824 (21.7%)
Productos en clase B: 964 (25.4%)
Productos en clase C: 2,009 (52.9%)

--- Referencia: conjunto completo (dic. 2009–dic. 2011) ---
Total de productos únicos: 4,725
Productos en clase A: 1,030
Proxy de ingresos del conjunto completo: £19,643,861.64


## 6. Cifras adicionales para el Business Case

Calculo el resto de las cifras que aparecen en el documento del Hito I: perfil del negocio, estacionalidad semanal y patrón de demanda de los productos. Todo sobre la ventana de 12 meses, salvo las cancelaciones, que se miden en el período completo.

In [8]:
# # 🟢 Perfil del negocio durante los doce meses analizados.
# facturas = df_12m.groupby("Invoice")["Revenue"].sum()  # 🟢 valor total por factura después de la limpieza
# n_facturas = len(facturas)
# mediana_factura = facturas.median()
# peso_facturas_500 = facturas[facturas >= 500].sum() / facturas.sum()

# ingresos_uk = df_12m.loc[df_12m["Country"] == "United Kingdom", "Revenue"].sum() / total_revenue_year
# n_paises = df_12m["Country"].nunique()

# # 🟢 Cancelaciones: se calcula el indicador para todo el período disponible.
# tasa_lineas_canc = len(df_canc) / n_filas_tras_dedup
# # 🔵 [IA] Aplico el mismo patrón de códigos a las cancelaciones para que la comparación sea coherente.
# mascara_canc_producto = (
#     df_canc["StockCode"].str.match(r"^\d{5}[A-Z]*$", na=False)
#     & (df_canc["Price"] > 0)
# )
# df_canc_producto = df_canc.loc[mascara_canc_producto].copy()
# valor_cancelado = -(df_canc_producto["Quantity"] * df_canc_producto["Price"]).sum()
# peso_canc_valor = valor_cancelado / ventas_brutas if ventas_brutas else np.nan

# # 🟢 Estacionalidad semanal durante los doce meses analizados.
# df_12m["Semana"] = df_12m["InvoiceDate"].dt.to_period("W-SUN")
# ventas_semana_completa = df_12m.groupby("Semana")["Revenue"].sum().iloc[1:-1]  # 🟢 excluyo las semanas parciales de los extremos
# pico_vs_mediana = ventas_semana_completa.max() / ventas_semana_completa.median()
# mes_pico_vs_promedio = monthly_revenue.max() / monthly_revenue.mean()
# print("Perfil y estacionalidad calculados.")

In [9]:
# 🟢 Perfil del negocio durante los doce meses analizados.
facturas = df_12m.groupby("Invoice")["Revenue"].sum()
n_facturas = len(facturas)
mediana_factura = facturas.median()
peso_facturas_500 = facturas[facturas >= 500].sum() / facturas.sum()

ingresos_uk = (
    df_12m.loc[
        df_12m["Country"] == "United Kingdom", "Revenue"
    ].sum() / total_revenue_year
)
n_paises = df_12m["Country"].nunique()

# 🟢 Cancelaciones: indicador para todo el período disponible.
tasa_lineas_canc = len(df_canc) / n_filas_tras_dedup

# 🟢 Aplicar a las cancelaciones la misma regla de elegibilidad
# que se utiliza para las ventas positivas en el Paso 4.
stockcode_canc_norm = (
    df_canc["StockCode"]
    .astype("string")
    .str.strip()
    .str.upper()
)

mascara_canc_codigo = (
    stockcode_canc_norm.str.match(
        patron_producto, na=False
    )
    | stockcode_canc_norm.isin(
        codigos_excepcion_producto
    )
)

# Mantener únicamente líneas con precio positivo para esta
# estimación descriptiva del valor de las cancelaciones.
mascara_canc_producto = (
    mascara_canc_codigo
    & df_canc["Price"].gt(0)
)

df_canc_producto = df_canc.loc[
    mascara_canc_producto
].copy()

# Las cantidades canceladas son negativas; se invierte el signo
# para presentar el valor cancelado como una cifra positiva.
valor_cancelado = -(
    df_canc_producto["Quantity"]
    * df_canc_producto["Price"]
).sum()

# Indicador descriptivo para todo el período, no una estimación
# de pérdidas reales ni de cancelaciones emparejadas con ventas.
peso_canc_valor = (
    valor_cancelado / ventas_brutas
    if ventas_brutas
    else np.nan
)

# 🟢 Estacionalidad semanal durante los doce meses analizados.
df_12m["Semana"] = df_12m["InvoiceDate"].dt.to_period("W-SUN")

ventas_semana_completa = (
    df_12m.groupby("Semana")["Revenue"].sum().iloc[1:-1]
)

# Se excluyen las semanas parciales de los extremos.
pico_vs_mediana = (
    ventas_semana_completa.max()
    / ventas_semana_completa.median()
)

mes_pico_vs_promedio = (
    monthly_revenue.max() / monthly_revenue.mean()
)

print("Perfil y estacionalidad calculados.")

Perfil y estacionalidad calculados.


### Patrón de demanda de cada producto (Syntetos-Boylan)

Para pronosticar importa **cómo** se vende un producto, no solo cuánto. Con las ventas semanales calculo dos medidas:

- **ADI:** número total de semanas del horizonte dividido entre el número de semanas con demanda positiva. Incluye semanas sin venta tanto al principio como al final.
- **CV²:** variabilidad del tamaño de la demanda positiva, elevada al cuadrado.

Con eso clasifico: *smooth* (frecuente y estable), *erratic* (frecuente pero muy variable), *intermittent* (ocasional y estable) y *lumpy* (ocasional y muy variable). Los umbrales usados son ADI = 1.32 y CV² = 0.49.


In [10]:
# 🔵 [IA] Creo una tabla por producto y semana; las semanas sin ventas se completan con cero para incluirlas en el ADI.
semanas = pd.period_range(df_12m["Semana"].min(), df_12m["Semana"].max(), freq="W-SUN")
unidades = (
    df_12m.groupby(["StockCode", "Semana"])["Quantity"].sum()
    .unstack(fill_value=0)
    .reindex(columns=semanas, fill_value=0)
)


def clasificar_demanda(fila):
    """Clasifica demanda semanal por SKU usando ADI y CV² sobre todo el horizonte observado."""
    con_venta = fila[fila > 0]
    if len(con_venta) < 2:
        return "insufficient"  # 🟢 hay muy pocas semanas con ventas para estimar el patrón
    adi = len(fila) / len(con_venta)  # 🔵 [IA] incluyo las semanas sin venta al principio y al final del horizonte
    cv2 = (con_venta.std(ddof=0) / con_venta.mean()) ** 2
    if adi < 1.32:
        return "smooth" if cv2 < 0.49 else "erratic"
    return "intermittent" if cv2 < 0.49 else "lumpy"


patron = unidades.apply(clasificar_demanda, axis=1)
patron_A = patron.reindex(prod_rev.loc[prod_rev["Clase"] == "A", "StockCode"]).dropna()
mix_A = patron_A.value_counts(normalize=True)

# 🟢 Número de semanas con ventas para cada producto de clase A.
semanas_activas_A = (unidades.reindex(patron_A.index) > 0).sum(axis=1)
print("Patrón de demanda calculado con ADI sobre el horizonte completo.")

Patrón de demanda calculado con ADI sobre el horizonte completo.


## 7. Resumen de cifras para el Business Case

Este bloque junta todas las cifras en un solo lugar para pasarlas al documento.

In [11]:
print("RESUMEN PARA EL BUSINESS CASE")
print("=" * 62)
print(f"Filas originales / limpias           : {initial_rows:,} / {len(df_clean):,}")
print(f"Ingresos 12 meses (dic10-nov11)      : £{total_revenue_year:,.2f}")
print(f"Productos (SKUs) vendidos 12 meses   : {len(prod_rev):,}")
print(f"Facturas 12 meses                    : {n_facturas:,}  (mediana £{mediana_factura:,.0f})")
print(f"Peso de facturas >= £500             : {peso_facturas_500:.1%} de los ingresos")
print(f"Reino Unido / países                 : {ingresos_uk:.1%} / {n_paises}")
print(f"Cancelaciones (período completo)     : {tasa_lineas_canc:.2%} de filas deduplicadas")
# print(f"Valor cancelado en SKU de patrón aceptado: £{valor_cancelado:,.0f} ({peso_canc_valor:.1%} de ventas brutas aceptadas, ratio descriptivo)")
print("Valor cancelado en SKU elegibles " "(patrón estándar + excepciones auditadas): "f"£{valor_cancelado:,.0f}")
print(f"Share sep-nov                        : {sep_nov_share:.1%}")
print(f"Semana pico / semana mediana         : {pico_vs_mediana:.2f}")
print(f"Mes pico / mes promedio              : {mes_pico_vs_promedio:.2f}  ({monthly_revenue.idxmax()})")
print(f"Clase A (80% ingresos)               : {class_a_products:,} SKUs ({class_a_products / len(prod_rev):.1%})")
print(f"Clase B / Clase C                    : {int(conteo['B']):,} ({conteo['B'] / len(prod_rev):.1%}) / {int(conteo['C']):,} ({conteo['C'] / len(prod_rev):.1%})")
print("Patrón de demanda en clase A         : " + ", ".join(f"{k} {v:.1%}" for k, v in mix_A.items()))
print(f"Mediana de semanas con venta (clase A): {semanas_activas_A.median():.0f} de {len(semanas)}")

RESUMEN PARA EL BUSINESS CASE
Filas originales / limpias           : 1,067,371 / 1,003,357
Ingresos 12 meses (dic10-nov11)      : £9,632,728.40
Productos (SKUs) vendidos 12 meses   : 3,797
Facturas 12 meses                    : 18,957  (mediana £302)
Peso de facturas >= £500             : 65.7% de los ingresos
Reino Unido / países                 : 84.6% / 38
Cancelaciones (período completo)     : 1.85% de filas deduplicadas
Valor cancelado en SKU elegibles (patrón estándar + excepciones auditadas): £716,463
Share sep-nov                        : 37.2%
Semana pico / semana mediana         : 2.20
Mes pico / mes promedio              : 1.81  (2011-11)
Clase A (80% ingresos)               : 824 SKUs (21.7%)
Clase B / Clase C                    : 964 (25.4%) / 2,009 (52.9%)
Patrón de demanda en clase A         : erratic 52.9%, lumpy 25.4%, smooth 12.5%, intermittent 9.2%
Mediana de semanas con venta (clase A): 49 de 53


## 8. Conclusiones y puntos para la bitácora

- El pipeline deduplica filas tras normalizar identificadores, separa cancelaciones, conserva ventas de cantidad y precio positivos y audita los códigos excluidos antes de aplicar el filtro inicial de SKU.
- **No se restan ventas por emparejamiento heurístico con cancelaciones.** El archivo no garantiza que una coincidencia de cliente/SKU/cantidad/precio identifique la línea original; los efectos de las devoluciones se reportan aparte hasta validar una llave confiable.
- La ventana dic-2010 a nov-2011 permite comparar 12 meses completos. Septiembre–noviembre representa cerca de **37%** del valor bruto de las ventas positivas aceptadas; esto demuestra estacionalidad descriptiva, no beneficio ni pérdida por falta de stock.
- Pocos productos concentran gran parte de los ingresos. La clase ABC debe usarse para priorizar el análisis, no como evidencia suficiente de que un pronóstico específico es mejor.
- La clasificación de patrón de demanda usa ADI calculado sobre todas las semanas, incluidas las semanas sin venta al comienzo y al final de la serie. Interpreta el resultado tras ejecutar el notebook y comparar los errores de los modelos en un backtest temporal.


### Auditoría de códigos de producto no estándar

In [12]:
import pandas as pd
import numpy as np

# Crear una copia independiente para auditar los códigos excluidos.
# Se conserva la regla actual de deduplicación sobre todas las columnas.
audit = df.loc[
    ~df.duplicated(),
    ["Invoice", "StockCode", "Description",
     "Quantity", "Price", "InvoiceDate"]
].copy()

# Reaplicar las reglas anteriores al filtro de códigos.
audit["Invoice"] = audit["Invoice"].astype(str).str.strip()
audit["StockCode"] = audit["StockCode"].astype(str).str.strip().str.upper()
audit["InvoiceDate"] = pd.to_datetime(
    audit["InvoiceDate"], errors="coerce"
)

audit = audit.loc[
    ~audit["Invoice"].str.startswith("C", na=False)
].copy()

audit = audit.loc[
    (audit["Quantity"] > 0) & (audit["Price"] > 0)
].copy()

audit["Revenue"] = audit["Quantity"] * audit["Price"]

# Identificar los códigos excluidos por la regla actual.
patron_producto = r"^\d{5}[A-Z]*$"
excluidos = audit.loc[
    ~audit["StockCode"].str.match(patron_producto, na=False)
].copy()

# Comparar el período anual utilizado en el Business Case.
en_periodo = (
    (excluidos["InvoiceDate"] >= "2010-12-01")
    & (excluidos["InvoiceDate"] < "2011-12-01")
)

excluidos["Filas_12m"] = en_periodo.astype(int)
excluidos["Ventas_12m"] = np.where(
    en_periodo, excluidos["Revenue"], 0.0
)

# Resumir impacto por código.
auditoria_codigos = (
    excluidos.groupby("StockCode", dropna=False)
    .agg(
        Filas_total=("StockCode", "size"),
        Descripcion=("Description", "first"),
        Valor_bruto_total=("Revenue", "sum"),
        Filas_12m=("Filas_12m", "sum"),
        Valor_bruto_12m=("Ventas_12m", "sum"),
    )
    .sort_values("Valor_bruto_12m", ascending=False)
)

print("Códigos excluidos distintos:", len(auditoria_codigos))
print("Filas positivas excluidas:", len(excluidos))
print(
    "Valor bruto positivo excluido en 12 meses: £{:,.2f}".format(
        excluidos["Ventas_12m"].sum()
    )
)
print("\nCódigos excluidos con mayor valor anual:")
display(auditoria_codigos.head(25).round(2))

Códigos excluidos distintos: 38
Filas positivas excluidas: 4699
Valor bruto positivo excluido en 12 meses: £371,987.23

Códigos excluidos con mayor valor anual:


,Filas_total,Descripcion,Valor_bruto_total,Filas_12m,Valor_bruto_12m
StockCode,,,,,
DOT,1415,DOTCOM POSTAGE,309854.11,690,186376.08
M,851,Manual,339241.29,309,77314.72
POST,1851,POSTAGE,125682.42,1076,75336.59
AMAZONFEE,3,AMAZON FEE,20467.80,2,13761.09
B,1,Adjust bad debt,11062.06,1,11062.06
C2,267,CARRIAGE,13426.00,137,6851.00
GIFT_0001_30,24,Dotcomgiftshop Gift Voucher £30.00,610.09,7,175.53
GIFT_0001_50,6,Dotcomgiftshop Gift Voucher £50.00,253.59,4,167.56
GIFT_0001_20,26,Dotcomgiftshop Gift Voucher £20.00,471.06,9,167.05


In [13]:
# Mostrar los 13 códigos restantes de la auditoría.
restantes = auditoria_codigos.iloc[25:].reset_index()

display(restantes.round(2))

,StockCode,Filas_total,Descripcion,Valor_bruto_total,Filas_12m,Valor_bruto_12m
0,DCGS0066N,2,NAVY CUDDLES DOG HOODIE,17.30,0,0.0
1,DCGS0041,1,HAYNES MINI-COOPER PLAYING CARDS,5.06,0,0.0
2,DCGS0044,1,HANDZ-OFF CAR FRESHENER,2.57,0,0.0
3,DCGS0058,30,MISO PRETTY GUM,34.82,0,0.0
4,DCGS0062,1,ROAD-RAGE CAR FRESHENER,2.51,0,0.0
5,DCGS0037,1,KEY-RING CORKSCREW,12.72,0,0.0
6,D,5,Discount,397.89,0,0.0
7,DCGS0068,2,DOGS NIGHT COLLAR,17.12,0,0.0
8,GIFT_0001_70,1,Dotcomgiftshop Gift Voucher £70.00,59.57,0,0.0
9,GIFT_0001_80,1,Dotcomgiftshop Gift Voucher £80.00,69.56,0,0.0


In [14]:
# Códigos no estándar que podrían representar mercancía física.
candidatos = [
    "DCGSSBOY",
    "DCGSSGIRL",
    "DCGS0076",
    "DCGS0004",
    "DCGS0069",
    "DCGS0070",
    "DCGS0003",
    "PADS",
]

# Revisar las transacciones dentro de la ventana anual.
periodo = excluidos.loc[
    excluidos["StockCode"].isin(candidatos)
    & (excluidos["InvoiceDate"] >= "2010-12-01")
    & (excluidos["InvoiceDate"] < "2011-12-01"),
    [
        "InvoiceDate",
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "Price",
        "Revenue",
    ],
].copy()

display(periodo.sort_values(["StockCode", "InvoiceDate"]))

print("Filas candidatas:", len(periodo))
print("Códigos candidatos:", periodo["StockCode"].nunique())
print(
    "Valor bruto exacto: £{:,.4f}".format(
        periodo["Revenue"].sum()
    )
)

,InvoiceDate,Invoice,StockCode,Description,Quantity,Price,Revenue
550367,2010-12-10 14:59:00,538349,DCGS0003,BOXED GLASS ASHTRAY,1,2.510,2.510
561921,2010-12-17 16:59:00,539451,DCGS0003,BOXED GLASS ASHTRAY,1,2.510,2.510
601712,2011-01-31 09:09:00,542622,DCGS0003,BOXED GLASS ASHTRAY,1,2.460,2.460
701467,2011-05-05 15:42:00,551995,DCGS0003,BOXED GLASS ASHTRAY,1,2.460,2.460
696244,2011-04-27 17:23:00,551340,DCGS0004,HAYNES CAMPER SHOULDER BAG,1,16.630,16.630
600514,2011-01-28 14:25:00,542541,DCGS0069,OOH LA LA DOGS COLLAR,1,15.790,15.790
565513,2010-12-21 13:06:00,539718,DCGS0070,CAMOUFLAGE DOG COLLAR,1,12.720,12.720
524271,2010-12-09 14:09:00,538071,DCGS0076,SUNJAR LED NIGHT NIGHT LIGHT,1,16.130,16.130
564774,2010-12-20 15:03:00,539631,DCGS0076,SUNJAR LED NIGHT NIGHT LIGHT,2,16.130,32.260
609477,2011-02-07 14:04:00,543358,DCGSSBOY,BOYS PARTY BAG,1,3.290,3.290


Filas candidatas: 35
Códigos candidatos: 8
Valor bruto exacto: £395.1630


# Diagnóstico previo a la agregación semanal de demanda

In [15]:
print("Shape del DataFrame:", df_clean.shape)
print("Columnas:", df_clean.columns.tolist())

print("\nTipos de datos:")
print(df_clean.dtypes)

print("\nCobertura temporal:")
print("Fecha inicial:", df_clean["InvoiceDate"].min())
print("Fecha final:", df_clean["InvoiceDate"].max())

print("\nCalidad de datos:")
print("Fechas ausentes:", df_clean["InvoiceDate"].isna().sum())
print("Códigos de producto ausentes:", df_clean["StockCode"].isna().sum())
print("Cantidades no positivas:", (df_clean["Quantity"] <= 0).sum())
print("Precios no positivos:", (df_clean["Price"] <= 0).sum())

print("\nResumen de demanda:")
print("SKUs distintos:", df_clean["StockCode"].nunique())
print("Unidades vendidas:", df_clean["Quantity"].sum())

Shape del DataFrame: (1003357, 9)
Columnas: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Revenue']

Tipos de datos:
Invoice                   str
StockCode                 str
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
Revenue               float64
dtype: object

Cobertura temporal:
Fecha inicial: 2009-12-01 07:45:00
Fecha final: 2011-12-09 12:50:00

Calidad de datos:
Fechas ausentes: 0
Códigos de producto ausentes: 0
Cantidades no positivas: 0
Precios no positivos: 0

Resumen de demanda:
SKUs distintos: 4725
Unidades vendidas: 11188061


## 9. Preparación de demanda semanal para pronóstico

🔵 [IA] Se construye una serie semanal por SKU utilizando
semanas calendario completas. Las semanas sin ventas
registradas se representan con cero, sin asumir que
necesariamente hubo demanda real nula.

In [16]:
# 🔵 [IA] Preparar una serie semanal de ventas por SKU.
# Se excluyen semanas incompletas de los extremos.

import pandas as pd

inicio = pd.Timestamp("2009-12-07")
fin_exclusivo = pd.Timestamp("2011-11-28")

ventas_modelo = df_clean.loc[
    (df_clean["InvoiceDate"] >= inicio)
    & (df_clean["InvoiceDate"] < fin_exclusivo)
].copy()

# Convertir cada fecha a su semana, comenzando en lunes.
ventas_modelo["Semana"] = (
    ventas_modelo["InvoiceDate"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

# Sumar unidades positivas vendidas por SKU y semana.
demanda_observada = (
    ventas_modelo
    .groupby(["StockCode", "Semana"], as_index=False)
    .agg(Demanda=("Quantity", "sum"))
)

# Construir todas las combinaciones de SKU y semana.
# Los ceros representan ausencia de ventas registradas.
semanas_modelo = pd.date_range(
    start=inicio,
    end=fin_exclusivo - pd.Timedelta(days=7),
    freq="W-MON"
)

skus_modelo = sorted(
    demanda_observada["StockCode"].unique()
)

indice_completo = pd.MultiIndex.from_product(
    [skus_modelo, semanas_modelo],
    names=["StockCode", "Semana"]
)

demanda_semanal = (
    demanda_observada
    .set_index(["StockCode", "Semana"])
    .reindex(indice_completo, fill_value=0)
    .reset_index()
    .sort_values(["StockCode", "Semana"])
    .reset_index(drop=True)
)

# Verificaciones básicas.
assert not demanda_semanal.duplicated(
    ["StockCode", "Semana"]
).any()

assert demanda_semanal["Demanda"].ge(0).all()

assert (
    demanda_semanal["Demanda"].sum()
    == ventas_modelo["Quantity"].sum()
)

print("SKUs:", len(skus_modelo))
print("Semanas:", len(semanas_modelo))
print("Registros SKU-semana:", len(demanda_semanal))
print("Unidades totales:", demanda_semanal["Demanda"].sum())
print("Semanas sin ventas registradas:",
      (demanda_semanal["Demanda"] == 0).sum())

display(demanda_semanal.head(10))

SKUs: 4691
Semanas: 103
Registros SKU-semana: 483173
Unidades totales: 10642073
Semanas sin ventas registradas: 293807


,StockCode,Semana,Demanda
0,10002,2009-12-07,24
1,10002,2009-12-14,37
2,10002,2009-12-21,14
3,10002,2009-12-28,0
4,10002,2010-01-04,15
5,10002,2010-01-11,191
6,10002,2010-01-18,41
7,10002,2010-01-25,44
8,10002,2010-02-01,16
9,10002,2010-02-08,43


In [17]:
# 🔵 [IA] Verificar la disponibilidad de PySpark en modo local.

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Reabastece360")
    .master("local[*]")
    .getOrCreate()
)

print("Versión Spark:", spark.version)
print("Modo de ejecución:", spark.sparkContext.master)

c:\Users\frank\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Versión Spark: 4.2.0
Modo de ejecución: local[*]


In [18]:
# 🔵 [IA] Transformación semanal con PySpark.
# Se parte de los datos previamente depurados y se valida
# que las unidades coincidan con el cálculo en Pandas.

from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, LongType
)

# Localizar la raíz del proyecto desde el notebook.
raiz = Path.cwd()
if raiz.name.lower() == "notebooks":
    raiz = raiz.parent

carpeta_salida = raiz / "data" / "processed"
carpeta_salida.mkdir(parents=True, exist_ok=True)

archivo_entrada = carpeta_salida / "ventas_limpias_spark.csv"

# Crear una entrada independiente para el motor Spark.
ventas_modelo[
    ["StockCode", "InvoiceDate", "Quantity"]
].to_csv(archivo_entrada, index=False)

# Definir explícitamente los tipos de datos.
esquema = StructType([
    StructField("StockCode", StringType(), False),
    StructField("InvoiceDate", TimestampType(), False),
    StructField("Quantity", LongType(), False),
])

ventas_spark = (
    spark.read
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .schema(esquema)
    .csv(str(archivo_entrada))
)

# Calcular la fecha del lunes de cada semana.
demanda_spark = (
    ventas_spark
    .withColumn(
        "Semana",
        F.to_date(F.date_trunc("week", F.col("InvoiceDate")))
    )
    .groupBy("StockCode", "Semana")
    .agg(F.sum("Quantity").alias("Demanda"))
)

# Validación de equivalencia básica.
total_pandas = int(demanda_observada["Demanda"].sum())
total_spark = int(
    demanda_spark.agg(F.sum("Demanda")).first()[0]
)

print("Unidades Pandas:", total_pandas)
print("Unidades Spark:", total_spark)

assert total_pandas == total_spark, (
    "Las unidades agregadas no coinciden."
)

print("Validación de unidades: CORRECTA")
print("Combinaciones SKU-semana Spark:", demanda_spark.count())

Unidades Pandas: 10642073
Unidades Spark: 10642073
Validación de unidades: CORRECTA
Combinaciones SKU-semana Spark: 189366


In [19]:
# 🔵 [IA] Validación exhaustiva entre Pandas y PySpark.
# Comparamos cada combinación de SKU y semana.

# 1. Verificar que Spark interpretó los datos correctamente.
filas_invalidas = ventas_spark.filter(
    F.col("StockCode").isNull()
    | F.col("InvoiceDate").isNull()
    | F.col("Quantity").isNull()
).count()

assert filas_invalidas == 0, (
    f"Spark encontró {filas_invalidas} filas inválidas."
)

# 2. Crear diccionario de referencia desde Pandas.
referencia = {
    (str(sku), fecha.date()): int(unidades)
    for sku, fecha, unidades in demanda_observada[
        ["StockCode", "Semana", "Demanda"]
    ].itertuples(index=False, name=None)
}

# 3. Recuperar resultados agregados de Spark.
resultado_spark = {}

for fila in demanda_spark.toLocalIterator():
    clave = (str(fila.StockCode), fila.Semana)
    assert clave not in resultado_spark, (
        f"Grupo duplicado: {clave}"
    )
    resultado_spark[clave] = int(fila.Demanda)

# 4. Comparar claves y cantidades.
assert referencia.keys() == resultado_spark.keys(), (
    "Existen diferencias entre los grupos SKU-semana."
)

diferencias = [
    clave
    for clave in referencia
    if referencia[clave] != resultado_spark[clave]
]

assert not diferencias, (
    f"Se encontraron {len(diferencias)} diferencias."
)

print("Filas inválidas en Spark:", filas_invalidas)
print("Grupos Pandas:", len(referencia))
print("Grupos Spark:", len(resultado_spark))
print("Diferencias:", len(diferencias))
print("VALIDACIÓN SKU-SEMANA: CORRECTA")

Filas inválidas en Spark: 0
Grupos Pandas: 189366
Grupos Spark: 189366
Diferencias: 0
VALIDACIÓN SKU-SEMANA: CORRECTA


In [20]:
# 🔵 [IA] Preparar las divisiones temporales y seleccionar
# productos utilizando únicamente datos históricos.

import numpy as np
import pandas as pd

semanas = sorted(demanda_semanal["Semana"].unique())

semanas_entrenamiento = semanas[:-24]
semanas_validacion = semanas[-24:-12]
semanas_prueba = semanas[-12:]

fecha_inicio_validacion = semanas_validacion[0]
fecha_inicio_prueba = semanas_prueba[0]

historial_seleccion = demanda_semanal.loc[
    demanda_semanal["Semana"] < fecha_inicio_validacion
]

actividad = historial_seleccion.groupby("StockCode").agg(
    semanas_positivas=("Demanda", lambda x: (x > 0).sum()),
    unidades_historicas=("Demanda", "sum")
)

skus_elegibles = actividad.loc[
    (actividad["semanas_positivas"] >= 12)
    & (actividad["unidades_historicas"] >= 30)
].index

panel = demanda_semanal.loc[
    demanda_semanal["StockCode"].isin(skus_elegibles)
].copy().sort_values(["StockCode", "Semana"])

print("SKUs elegibles:", len(skus_elegibles))
print("Semanas de entrenamiento:", len(semanas_entrenamiento))
print("Semanas de validación:", len(semanas_validacion))
print("Semanas de prueba:", len(semanas_prueba))
print("Inicio validación:", fecha_inicio_validacion)
print("Inicio prueba:", fecha_inicio_prueba)

SKUs elegibles: 3135
Semanas de entrenamiento: 79
Semanas de validación: 12
Semanas de prueba: 12
Inicio validación: 2011-06-13 00:00:00
Inicio prueba: 2011-09-05 00:00:00


In [21]:
# 🔵 [IA] Crear pronósticos de referencia sin fuga temporal.
# shift(1) evita utilizar la demanda de la misma semana
# que se intenta predecir.

grupo = panel.groupby("StockCode")["Demanda"]

panel["pronostico_cero"] = 0.0
panel["pronostico_ultima_semana"] = grupo.shift(1)

panel["pronostico_media_4"] = (
    panel.groupby("StockCode")["Demanda"]
    .transform(
        lambda serie: (
            serie.shift(1)
            .rolling(window=4, min_periods=4)
            .mean()
        )
    )
)

# Evaluar todos los modelos sobre las mismas observaciones.
def evaluar_pronostico(datos, columna):
    reales = datos["Demanda"].to_numpy(dtype=float)
    predicciones = datos[columna].to_numpy(dtype=float)

    mae = np.mean(np.abs(reales - predicciones))
    rmse = np.sqrt(np.mean((reales - predicciones) ** 2))
    denominador = np.sum(np.abs(reales))

    wape = (
        100 * np.sum(np.abs(reales - predicciones)) / denominador
        if denominador > 0 else np.nan
    )

    sesgo = np.mean(predicciones - reales)

    return {
        "Modelo": columna,
        "MAE": mae,
        "RMSE": rmse,
        "WAPE_pct": wape,
        "Sesgo": sesgo
    }

columnas = [
    "pronostico_cero",
    "pronostico_ultima_semana",
    "pronostico_media_4"
]

resultados = []

for nombre, inicio, fin in [
    ("Validación", fecha_inicio_validacion, fecha_inicio_prueba),
    ("Prueba", fecha_inicio_prueba, None)
]:
    mascara = panel["Semana"] >= inicio
    if fin is not None:
        mascara &= panel["Semana"] < fin

    datos = panel.loc[mascara].dropna(subset=columnas)

    for columna in columnas:
        resultado = evaluar_pronostico(datos, columna)
        resultado["Periodo"] = nombre
        resultados.append(resultado)

tabla_baselines = pd.DataFrame(resultados)

display(tabla_baselines.round(3))

,Modelo,MAE,RMSE,WAPE_pct,Sesgo,Periodo
0,pronostico_cero,23.518,85.767,100.000,-23.518,Validación
1,pronostico_ultima_semana,21.570,83.018,91.717,0.617,Validación
2,pronostico_media_4,18.610,66.742,79.131,0.172,Validación
3,pronostico_cero,34.910,120.195,100.000,-34.910,Prueba
4,pronostico_ultima_semana,27.174,100.977,77.841,-1.397,Prueba
5,pronostico_media_4,23.345,82.314,66.872,-3.580,Prueba


In [22]:
# Preparar las variables históricas para el pronóstico semanal.
# Ordenamos los datos para respetar la secuencia temporal de cada producto.

datos_ml = panel.copy().sort_values(
    ["StockCode", "Semana"]
)

# Agrupar las cantidades vendidas por producto.
grupo = datos_ml.groupby("StockCode")["Demanda"]

# Recuperar la demanda registrada en la semana anterior.
datos_ml["lag_1"] = grupo.shift(1)

# Revisar los resultados.
display(
    datos_ml[
        ["StockCode", "Semana", "Demanda", "lag_1"]
    ].head(10)
)

,StockCode,Semana,Demanda,lag_1
0,10002,2009-12-07,24,NaN
1,10002,2009-12-14,37,24.0
2,10002,2009-12-21,14,37.0
3,10002,2009-12-28,0,14.0
4,10002,2010-01-04,15,0.0
5,10002,2010-01-11,191,15.0
6,10002,2010-01-18,41,191.0
7,10002,2010-01-25,44,41.0
8,10002,2010-02-01,16,44.0
9,10002,2010-02-08,43,16.0


In [23]:
datos_ml.groupby("StockCode")["Demanda"].shift(1)

0          NaN
1         24.0
2         37.0
3         14.0
4          0.0
          ... 
482859     0.0
482860     0.0
482861     0.0
482862     0.0
482863     0.0
Name: Demanda, Length: 322905, dtype: float64

In [24]:
datos_ml["lag_1"] = grupo.shift(1)

# Demanda observada hace dos semanas.
datos_ml["lag_2"] = grupo.shift(2)

# Demanda observada hace cuatro semanas.
datos_ml["lag_4"] = grupo.shift(4)

In [25]:
display(
    datos_ml[
        ["StockCode", "Semana", "Demanda",
         "lag_1", "lag_2", "lag_4"]
    ].head(10)
)

,StockCode,Semana,Demanda,lag_1,lag_2,lag_4
0,10002,2009-12-07,24,NaN,NaN,NaN
1,10002,2009-12-14,37,24.0,NaN,NaN
2,10002,2009-12-21,14,37.0,24.0,NaN
3,10002,2009-12-28,0,14.0,37.0,NaN
4,10002,2010-01-04,15,0.0,14.0,24.0
5,10002,2010-01-11,191,15.0,0.0,37.0
6,10002,2010-01-18,41,191.0,15.0,14.0
7,10002,2010-01-25,44,41.0,191.0,0.0
8,10002,2010-02-01,16,44.0,41.0,15.0
9,10002,2010-02-08,43,16.0,44.0,191.0


In [26]:
# Calcular la demanda promedio de las cuatro semanas anteriores.
datos_ml["media_4"] = (
    datos_ml.groupby("StockCode")["Demanda"]
    .transform(
        lambda serie: serie.shift(1)
        .rolling(window=4, min_periods=4)
        .mean()
    )
)

# Revisar las primeras observaciones.
display(
    datos_ml[
        ["StockCode", "Semana", "Demanda", "media_4"]
    ].head(10)
)

,StockCode,Semana,Demanda,media_4
0,10002,2009-12-07,24,NaN
1,10002,2009-12-14,37,NaN
2,10002,2009-12-21,14,NaN
3,10002,2009-12-28,0,NaN
4,10002,2010-01-04,15,18.75
5,10002,2010-01-11,191,16.50
6,10002,2010-01-18,41,55.00
7,10002,2010-01-25,44,61.75
8,10002,2010-02-01,16,72.75
9,10002,2010-02-08,43,73.00


In [27]:
# Completar las variables históricas del modelo.
# Cada variable utiliza información anterior a la semana objetivo.

datos_ml["lag_8"] = grupo.shift(8)

datos_ml["media_8"] = (
    datos_ml.groupby("StockCode")["Demanda"]
    .transform(
        lambda serie: serie.shift(1)
        .rolling(window=8, min_periods=8)
        .mean()
    )
)

# Representar la posición de la semana dentro del año.
# Las funciones seno y coseno permiten representar
# la continuidad entre el final y el inicio del año.

semana_anio = datos_ml["Semana"].dt.isocalendar().week.astype(int)

datos_ml["semana_sen"] = np.sin(
    2 * np.pi * semana_anio / 52.18
)

datos_ml["semana_cos"] = np.cos(
    2 * np.pi * semana_anio / 52.18
)

variables = [
    "lag_1", "lag_2", "lag_4", "lag_8",
    "media_4", "media_8",
    "semana_sen", "semana_cos"
]

# Comprobar las variables generadas.
print(datos_ml[variables].isna().sum())

display(datos_ml[["StockCode", "Semana"] + variables].head(12))

lag_1          3135
lag_2          6270
lag_4         12540
lag_8         25080
media_4       12540
media_8       25080
semana_sen        0
semana_cos        0
dtype: int64


,StockCode,Semana,lag_1,lag_2,lag_4,lag_8,media_4,media_8,semana_sen,semana_cos
0,10002,2009-12-07,NaN,NaN,NaN,NaN,NaN,NaN,-0.259497,0.965744
1,10002,2009-12-14,24.0,NaN,NaN,NaN,NaN,NaN,-0.141611,0.989922
2,10002,2009-12-21,37.0,24.0,NaN,NaN,NaN,NaN,-0.021673,0.999765
3,10002,2009-12-28,14.0,37.0,NaN,NaN,NaN,NaN,0.098579,0.995129
4,10002,2010-01-04,0.0,14.0,24.0,NaN,18.75,NaN,0.120123,0.992759
5,10002,2010-01-11,15.0,0.0,37.0,NaN,16.50,NaN,0.238506,0.971141
6,10002,2010-01-18,191.0,15.0,14.0,NaN,55.00,NaN,0.353435,0.935459
7,10002,2010-01-25,41.0,191.0,0.0,NaN,61.75,NaN,0.463246,0.886230
8,10002,2010-02-01,44.0,41.0,15.0,24.0,72.75,45.750,0.566348,0.824166
9,10002,2010-02-08,16.0,44.0,191.0,37.0,73.00,44.750,0.661249,0.750167


In [28]:
# Preparar datos para entrenamiento y validación.
# Se conservan únicamente observaciones con historial suficiente.

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

datos_entrenables = datos_ml.dropna(
    subset=variables
).copy()

entrenamiento = datos_entrenables.loc[
    datos_entrenables["Semana"] < fecha_inicio_validacion
].copy()

validacion = datos_entrenables.loc[
    (datos_entrenables["Semana"] >= fecha_inicio_validacion)
    & (datos_entrenables["Semana"] < fecha_inicio_prueba)
].copy()

X_train = entrenamiento[variables]
y_train = entrenamiento["Demanda"]

X_val = validacion[variables]
y_val = validacion["Demanda"]

print("Observaciones entrenamiento:", len(X_train))
print("Observaciones validación:", len(X_val))

# Modelos candidatos con parámetros iniciales fijos.
modelos = {
    "Ridge": make_pipeline(
        StandardScaler(),
        Ridge(alpha=100.0)
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=100,
        max_leaf_nodes=15,
        learning_rate=0.05,
        l2_regularization=10.0,
        random_state=42
    )
}

resultados_ml = []
modelos_entrenados = {}

for nombre, modelo in modelos.items():
    print(f"Entrenando {nombre}...")

    modelo.fit(X_train, y_train)
    modelos_entrenados[nombre] = modelo

    # La demanda prevista no puede ser negativa.
    predicciones = np.maximum(modelo.predict(X_val), 0)

    temporal = validacion[["Demanda"]].copy()
    temporal["prediccion"] = predicciones

    metricas = evaluar_pronostico(temporal, "prediccion")
    metricas["Modelo"] = nombre
    metricas["Periodo"] = "Validación"

    resultados_ml.append(metricas)

tabla_ml = pd.DataFrame(resultados_ml)

display(tabla_ml.round(3))

Observaciones entrenamiento: 222585
Observaciones validación: 37620
Entrenando Ridge...
Entrenando HistGradientBoosting...


,Modelo,MAE,RMSE,WAPE_pct,Sesgo,Periodo
0,Ridge,24.074,63.855,102.362,5.352,Validación
1,HistGradientBoosting,19.252,62.304,81.858,1.859,Validación


In [29]:
# Comparar los modelos ML con el promedio móvil de cuatro semanas.
# Todos se evalúan sobre el mismo período y productos.

referencia = tabla_baselines.loc[
    (tabla_baselines["Periodo"] == "Validación")
    & (tabla_baselines["Modelo"] == "pronostico_media_4")
]

comparacion = pd.concat(
    [referencia, tabla_ml],
    ignore_index=True
)

comparacion = comparacion.sort_values("WAPE_pct")

display(
    comparacion[
        ["Modelo", "MAE", "RMSE", "WAPE_pct", "Sesgo"]
    ].round(3)
)

,Modelo,MAE,RMSE,WAPE_pct,Sesgo
0,pronostico_media_4,18.610,66.742,79.131,0.172
2,HistGradientBoosting,19.252,62.304,81.858,1.859
1,Ridge,24.074,63.855,102.362,5.352


In [30]:
# Comprobar que los modelos se evaluaron sobre los mismos registros.

validacion_referencia = panel.loc[
    (panel["Semana"] >= fecha_inicio_validacion)
    & (panel["Semana"] < fecha_inicio_prueba)
].copy()

columnas_comparacion = [
    "StockCode", "Semana", "Demanda"
]

referencia_ordenada = (
    validacion_referencia[columnas_comparacion]
    .sort_values(["StockCode", "Semana"])
    .reset_index(drop=True)
)

ml_ordenada = (
    validacion[columnas_comparacion]
    .sort_values(["StockCode", "Semana"])
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    referencia_ordenada,
    ml_ordenada
)

assert len(validacion) == 3135 * 12

print("Observaciones comparadas:", len(validacion))
print("Mismos SKU, semanas y demanda: CORRECTO")

Observaciones comparadas: 37620
Mismos SKU, semanas y demanda: CORRECTO


In [31]:
# Tercer modelo: regresión por vecinos cercanos.
# Se utilizan las mismas variables y fechas que los demás modelos.

from sklearn.neighbors import KNeighborsRegressor

modelo_knn = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(
        n_neighbors=15,
        weights="distance",
        n_jobs=-1
    )
)

print("Entrenando KNN...")

modelo_knn.fit(X_train, y_train)

predicciones_knn = np.maximum(
    modelo_knn.predict(X_val), 0
)

evaluacion_knn = validacion[["Demanda"]].copy()
evaluacion_knn["prediccion"] = predicciones_knn

metricas_knn = evaluar_pronostico(
    evaluacion_knn,
    "prediccion"
)

metricas_knn["Modelo"] = "KNN"
metricas_knn["Periodo"] = "Validación"

tabla_ml = pd.concat(
    [tabla_ml, pd.DataFrame([metricas_knn])],
    ignore_index=True
)

modelos_entrenados["KNN"] = modelo_knn

comparacion_final = pd.concat(
    [
        tabla_baselines.loc[
            (tabla_baselines["Periodo"] == "Validación")
            & (tabla_baselines["Modelo"] == "pronostico_media_4")
        ],
        tabla_ml
    ],
    ignore_index=True
).sort_values("WAPE_pct")

display(
    comparacion_final[
        ["Modelo", "MAE", "RMSE", "WAPE_pct", "Sesgo"]
    ].round(3)
)

Entrenando KNN...


,Modelo,MAE,RMSE,WAPE_pct,Sesgo
0,pronostico_media_4,18.610,66.742,79.131,0.172
2,HistGradientBoosting,19.252,62.304,81.858,1.859
3,KNN,20.530,70.327,87.295,1.539
1,Ridge,24.074,63.855,102.362,5.352


In [32]:
# Guardar los resultados de validación para documentar el experimento.

from pathlib import Path

carpeta_reportes = raiz / "reports"
carpeta_reportes.mkdir(parents=True, exist_ok=True)

ruta_metricas = carpeta_reportes / "comparacion_modelos_validacion.csv"

comparacion_final.to_csv(
    ruta_metricas,
    index=False,
    encoding="utf-8-sig"
)

# Conservar los pronósticos de la política de referencia seleccionada.
pronosticos_validacion = panel.loc[
    (panel["Semana"] >= fecha_inicio_validacion)
    & (panel["Semana"] < fecha_inicio_prueba),
    [
        "StockCode",
        "Semana",
        "Demanda",
        "pronostico_media_4"
    ]
].copy()

pronosticos_validacion = pronosticos_validacion.rename(
    columns={"pronostico_media_4": "Pronostico"}
)

assert not pronosticos_validacion["Pronostico"].isna().any()
assert pronosticos_validacion["Pronostico"].ge(0).all()

ruta_pronosticos = (
    carpeta_reportes / "pronosticos_validacion.csv"
)

pronosticos_validacion.to_csv(
    ruta_pronosticos,
    index=False,
    encoding="utf-8-sig"
)

print("Métricas guardadas:", ruta_metricas)
print("Pronósticos guardados:", ruta_pronosticos)
print("Registros:", len(pronosticos_validacion))

Métricas guardadas: c:\Users\frank\Reabastece360\reports\comparacion_modelos_validacion.csv
Pronósticos guardados: c:\Users\frank\Reabastece360\reports\pronosticos_validacion.csv
Registros: 37620


In [33]:
# Importar las funciones de almacenamiento del proyecto.

import sys

if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.almacenamiento.mongo_repo import (
    preparar_coleccion,
    guardar_pronosticos,
    consultar_pronosticos
)

print("Módulo de almacenamiento importado correctamente.")

Módulo de almacenamiento importado correctamente.


In [34]:
# Preparar los pronósticos históricos para almacenarlos.
# Cada fila se convierte en un documento con su SKU, semana y pronóstico.

documentos_pronosticos = []

for fila in pronosticos_validacion.itertuples(index=False):
    documento = {
        "sku": str(fila.StockCode),
        "semana": fila.Semana.strftime("%Y-%m-%d"),
        "modelo": "media_movil_4",
        "pronostico_unidades": float(fila.Pronostico),
        "demanda_real": int(fila.Demanda),
        "tipo_resultado": "validacion_historica"
    }

    documentos_pronosticos.append(documento)

print("Documentos preparados:", len(documentos_pronosticos))
print("Ejemplo del primer documento:")
print(documentos_pronosticos[0])

Documentos preparados: 37620
Ejemplo del primer documento:
{'sku': '10002', 'semana': '2011-06-13', 'modelo': 'media_movil_4', 'pronostico_unidades': 0.0, 'demanda_real': 0, 'tipo_resultado': 'validacion_historica'}


In [35]:
print("Documentos preparados:", len(documentos_pronosticos))
print("Primer documento:", documentos_pronosticos[0])

Documentos preparados: 37620
Primer documento: {'sku': '10002', 'semana': '2011-06-13', 'modelo': 'media_movil_4', 'pronostico_unidades': 0.0, 'demanda_real': 0, 'tipo_resultado': 'validacion_historica'}


In [36]:
# Almacenar los pronósticos históricos en una base de datos simulada.
# MongoDB real no se utiliza en esta prueba.

import mongomock

cliente = mongomock.MongoClient()
base_datos = cliente["reabastece360"]

coleccion = preparar_coleccion(base_datos)

# Utilizar una muestra de 100 pronósticos históricos.
muestra_pronosticos = documentos_pronosticos[:100]

guardar_pronosticos(
    coleccion,
    muestra_pronosticos
)

total_guardado = coleccion.count_documents({})

print("Documentos almacenados:", total_guardado)

assert total_guardado == len(muestra_pronosticos)

print("VALIDACIÓN DE ALMACENAMIENTO: CORRECTA")

Documentos almacenados: 100
VALIDACIÓN DE ALMACENAMIENTO: CORRECTA


In [37]:
# Simular una recomendación de reposición para un producto.
# El inventario, plazo de entrega y factor de seguridad son supuestos.

import math

ejemplo = pronosticos_validacion.loc[
    pronosticos_validacion["Pronostico"] > 0
].iloc[0]

sku = ejemplo["StockCode"]
pronostico_semanal = float(ejemplo["Pronostico"])

# Parámetros hipotéticos para demostrar la política.
plazo_entrega_semanas = 2
inventario_disponible = 100
factor_seguridad = 0.25

# Demanda estimada durante el plazo de entrega.
demanda_plazo = pronostico_semanal * plazo_entrega_semanas

# Reserva de seguridad simplificada.
stock_seguridad = demanda_plazo * factor_seguridad

# Nivel de inventario que activa una revisión de reposición.
# Redondear hacia arriba para trabajar con unidades enteras.
punto_reposicion = math.ceil(
    demanda_plazo + stock_seguridad
)

reponer = inventario_disponible <= punto_reposicion

print("SKU:", sku)
print("Pronóstico semanal:", round(pronostico_semanal, 2))
print("Demanda durante plazo de entrega:", round(demanda_plazo, 2))
print("Stock de seguridad:", round(stock_seguridad, 2))
print("Punto de reposición:", round(punto_reposicion, 2))
print("Inventario supuesto:", inventario_disponible)
print("¿Revisar reposición?:", "Sí" if reponer else "No")

SKU: 10120
Pronóstico semanal: 0.5
Demanda durante plazo de entrega: 1.0
Stock de seguridad: 0.25
Punto de reposición: 2
Inventario supuesto: 100
¿Revisar reposición?: No


## 10. Evaluación económica y financiera

Esta sección reproduce y revisa los escenarios financieros
propuestos en el Business Case del HITO I.

Los costos, beneficios y parámetros de inventario utilizados
son supuestos de planificación, no resultados operativos
observados en el conjunto de datos.

El análisis contempla escenarios conservador, base y favorable,
con horizontes de evaluación de 12 y 24 meses.

In [38]:
# Costos estimados del proyecto según el Business Case.
# Los importes son supuestos de planificación expresados en GBP.

costo_implementacion = 6400
costo_operacion_anual = 2640

costo_total_12m = (
    costo_implementacion + costo_operacion_anual
)

costo_total_24m = (
    costo_implementacion + 2 * costo_operacion_anual
)

print("Costo inicial: £", costo_implementacion)
print("Costo operativo anual: £", costo_operacion_anual)
print("Costo total a 12 meses: £", costo_total_12m)
print("Costo total a 24 meses: £", costo_total_24m)

Costo inicial: £ 6400
Costo operativo anual: £ 2640
Costo total a 12 meses: £ 9040
Costo total a 24 meses: £ 11680


In [39]:
# Supuestos de beneficios del Business Case.
# Ninguno representa ahorros reales comprobados.

ventas_anuales = 9632728.40
inventario_promedio = 200000
costo_mantenimiento = 0.25
valor_hora = 40

escenarios = {
    "Conservador": {
        "recuperacion": 0.10,
        "margen": 0.25,
        "reduccion_inventario": 0.05,
        "horas_ahorradas": 40,
        "costo_mantenimiento": 0.20
    },
    "Base": {
        "recuperacion": 0.25,
        "margen": 0.30,
        "reduccion_inventario": 0.10,
        "horas_ahorradas": 60,
        "costo_mantenimiento": 0.25
    },
    "Favorable": {
        "recuperacion": 0.30,
        "margen": 0.35,
        "reduccion_inventario": 0.12,
        "horas_ahorradas": 80,
        "costo_mantenimiento": 0.30
    }
}

print("Escenarios definidos:", list(escenarios.keys()))

Escenarios definidos: ['Conservador', 'Base', 'Favorable']


In [40]:
# Calcular los beneficios anuales hipotéticos de cada escenario.
# Se utilizan los supuestos establecidos en el Business Case.

ventas_en_riesgo_pct = 0.04
resultados_financieros = []

for nombre, supuesto in escenarios.items():

    # Contribución estimada de ventas recuperadas.
    beneficio_ventas = (
        ventas_anuales
        * ventas_en_riesgo_pct
        * supuesto["recuperacion"]
        * supuesto["margen"]
    )

    # Ahorro hipotético por reducir inventario promedio.
    beneficio_inventario = (
        inventario_promedio
        * supuesto["reduccion_inventario"]
        * supuesto["costo_mantenimiento"]
    )

    # Valor estimado del tiempo de trabajo ahorrado.
    beneficio_tiempo = (
        supuesto["horas_ahorradas"]
        * valor_hora
    )

    beneficio_anual = (
        beneficio_ventas
        + beneficio_inventario
        + beneficio_tiempo
    )

    resultados_financieros.append({
        "Escenario": nombre,
        "Ventas recuperadas": beneficio_ventas,
        "Ahorro inventario": beneficio_inventario,
        "Ahorro tiempo": beneficio_tiempo,
        "Beneficio anual": beneficio_anual
    })

tabla_beneficios = pd.DataFrame(resultados_financieros)

display(tabla_beneficios.round(2))

,Escenario,Ventas recuperadas,Ahorro inventario,Ahorro tiempo,Beneficio anual
0,Conservador,9632.73,2000.0,1600,13232.73
1,Base,28898.19,5000.0,2400,36298.19
2,Favorable,40457.46,7200.0,3200,50857.46


In [41]:
# Evaluar flujos de caja simulados a 12 y 24 meses.
# Se asumen beneficios anuales constantes y costos operativos uniformes.

flujos_financieros = []

for _, fila in tabla_beneficios.iterrows():

    beneficio = fila["Beneficio anual"]

    # Primer año: incluye el costo inicial de implementación.
    flujo_anio_1 = (
        beneficio
        - costo_operacion_anual
        - costo_implementacion
    )

    # Segundo año: solo se descuenta el costo operativo.
    flujo_anio_2 = beneficio - costo_operacion_anual

    beneficio_neto_24m = flujo_anio_1 + flujo_anio_2

    roi_24m = (
        beneficio_neto_24m / costo_total_24m
    ) * 100

    flujos_financieros.append({
        "Escenario": fila["Escenario"],
        "Flujo neto 12m": flujo_anio_1,
        "Flujo neto año 2": flujo_anio_2,
        "Neto acumulado 24m": beneficio_neto_24m,
        "ROI 24m (%)": roi_24m
    })

tabla_flujos = pd.DataFrame(flujos_financieros)

display(tabla_flujos.round(2))

,Escenario,Flujo neto 12m,Flujo neto año 2,Neto acumulado 24m,ROI 24m (%)
0,Conservador,4192.73,10592.73,14785.46,126.59
1,Base,27258.19,33658.19,60916.37,521.54
2,Favorable,41817.46,48217.46,90034.92,770.85


In [42]:
# Guardar los escenarios financieros para el informe HITO II.

ruta_finanzas = carpeta_reportes / "evaluacion_financiera_24m.csv"

tabla_flujos.to_csv(
    ruta_finanzas,
    index=False,
    encoding="utf-8-sig"
)

print("Evaluación financiera guardada:", ruta_finanzas)
print("Escenarios exportados:", len(tabla_flujos))

Evaluación financiera guardada: c:\Users\frank\Reabastece360\reports\evaluacion_financiera_24m.csv
Escenarios exportados: 3


In [43]:
# Calcular el beneficio mínimo necesario para recuperar
# los costos del proyecto durante 24 meses.

beneficio_minimo_anual = costo_total_24m / 2

beneficio_base_anual = tabla_beneficios.loc[
    tabla_beneficios["Escenario"] == "Base",
    "Beneficio anual"
].iloc[0]

porcentaje_necesario = (
    beneficio_minimo_anual / beneficio_base_anual
) * 100

print("Beneficio mínimo anual: £", round(beneficio_minimo_anual, 2))
print("Porcentaje del escenario base necesario:", round(porcentaje_necesario, 2), "%")

Beneficio mínimo anual: £ 5840.0
Porcentaje del escenario base necesario: 16.09 %
